# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"\nName: {getattr(dataset.metadata, 'name', '')}\nDescription: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by @id, label, and description
print("Available record sets:")
record_sets = list(dataset.metadata.record_sets)
for record_set in record_sets:
    print(f" @id: {record_set['@id']}, name: {record_set.get('name', '')}, description: {record_set.get('description', '')}")
    # List fields (columns) in this record set
    print("   Fields in this record set:")
    for field in record_set.get('fields', []):
        print(f"    - @id: {field['@id']}, name: {field.get('name','')}, description: {field.get('description','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's get all record set @id values
record_set_ids = [r['@id'] for r in dataset.metadata.record_sets]
print("RecordSet @ids:", record_set_ids)

# Load each record set as a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded {len(df)} records for RecordSet: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")

# Choose the main (first) record set for further exploration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nPreview of first 5 rows from main record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note:** Make sure to use the correct `@id` of the numeric field and an appropriate grouping field from the overview above.

In [ ]:
# For demonstration, auto-select first numeric-looking and groupable fields based on column names
import numpy as np

df = dataframes[main_record_set_id]

# Try to infer some likely numeric and group fields
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Heuristic to pick numeric field (could be improved if schema known)
    if numeric_field_id is None and df[col].dtype in [np.float64, np.int64] or col.lower().endswith(('count', 'intensity', 'number', 'value', 'abundance')):
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            continue
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
    # Heuristic to pick group field
    if group_field_id is None and not np.issubdtype(df[col].dtype, np.number):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break
print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} records")

    # Normalize the numeric field
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print("First 5 normalized rows:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
    
    # Group by group_field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count']).reset_index()
        print(f"Grouped by {group_field_id}:")
        display(grouped.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        top10 = filtered_df[group_field_id].value_counts().index[:10]
        sns.boxplot(data=filtered_df[filtered_df[group_field_id].isin(top10)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric and/or group field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_This notebook provided an initial exploration of the Mass Spectrometry dataset using the `mlcroissant` library. We demonstrated how to load metadata, inspect record sets and fields, extract record data by `@id`, and perform basic exploratory data analysis and visualization using inferred numeric and grouping fields. For deeper analysis, users are encouraged to review the dataset schema for detailed field descriptions and adapt selection and processing accordingly._